In [1]:
from IPython.core.display import HTML
import pandas as pd
import numpy as np
import importlib as imp
import help_tools as ht
HTML("<style>.container { width: 120% !important; }</style>")
import warnings
warnings.filterwarnings('ignore')

In [2]:
client_id = '<SPOTIFY_CLIENT_ID>'
client_secret = '<SPOTIFY_CLIENT_SECRET>'
api_key_tmdb = '<TMDB_API_KEY>'
api_key_omdb = '<OMDB_API_KEY>'

In [3]:
# откуда накидывать id людей
# https://www.themoviedb.org/person/7030-henry-goodman

actor_dict = {
            'joaquin-phoenix' : 73421,
            'colin-farrell' : 72466,
            'leonardo-dicaprio' : 6193,
            'javier-bardem' : 3810,
            'anthony-hopkins' : 4173,
            'brad-pitt' : 287,
            'antonio-banderas': 3131,
            'matthew-mcconaughey' : 10297,
            'steve-buscemi' : 884,
            'tim-roth' : 3129,
            'mads-mikkelsen' : 1019,
            'scarlett-johansson' : 1245,
            'til-schweiger' : 1844,
            'george-clooney' : 1461,
            'tom-hanks' : 31,
            'robert-de-niro' : 380,
            'adrien-brody' : 3490,
            'matt-damon' : 1892,
            'edward-norton' : 819,
            'robert-pattinson' : 11288,
            'jude-law' : 9642,
            'cillian-murphy' : 2037,
            'benicio-del-toro' : 1121,
            'timothee-chalamet' : 1190668,

            'christian-bale': 3894,         # Интенсивность, метод (как Феникс/Мерфи)
            'adam-driver': 1023139,        # Работа с лучшими режиссерами (как Шаламе/Питт)
            'jake-gyllenhaal': 131,        # Психологические триллеры (как Джилленхол/Фаррелл)
            'willem-dafoe': 5293,          # Характерные роли (как Бушеми/Дель Торо)
            'paul-mescal': 2590209,        # Новая волна драмы (как Шаламе/Паттинсон)
            'michael-fassbender': 17288,   # Холодная экспрессия (как Миккельсен)
            'oscar-isaac': 25072,          # Интеллектуальное кино (как Броуди/Лоу)
            'barry-keoghan': 1290466,      # Специфическая органика (как Кеоган/Фаррелл)
            'jesse-plemons': 40685,        # Мастер второго плана (как Бушеми/Хоффман)
            'tom-hardy': 2524
             }

director_dict = {
                'quentin-tarantino' : 138,
                'david-fincher' : 7467,
                'jim-jarmusch' : 4429,
                'guy-ritchie' : 956,
                'christopher-nolan' : 525,
                'darren-aronofsky' : 6431,
                'martin-mcdonagh' : 54472,
                'pedro-almodovar' : 309,
                'paolo-sorrentino' : 56194,
                'cristian-mungiu' : 20657,
                'yorgos-lanthimos' : 122423,
                'bong-joon-ho' : 21684,
                'lars-von-trier' : 42,
                'woody-allen' : 1243,
                'denis-villeneuve': 137427,
                'wes-anderson': 5655,
                'paul-thomas-anderson': 3223,  # Психологизм (как Соррентино/Джармуш)
                'robert-eggers': 1313360,      # Атмосферный хоррор/драма (как Аронофски)
                'ari-aster': 1030513,          # Сюрреализм и травма (как Лантимос/Фон Триер)
                'park-chan-wook': 12453,       # Эстетика насилия (как Тарантино/Пон Чжун Хо)
                'jonathan-glazer': 53517,      # Радикальный формализм (как Мунджиу/Лантимос)
                'ruben-ostlund': 76043,        # Социальная сатира (как Лантимос/Макдона)
                'gaspar-noe': 2617,            # Провокация (как Фон Триер)
                'nicolas-winding-refn': 11252  # Неоновый стиль (как Финчер/Соррентино)
}

# фильмы которые не хочется чекать (уже просмотренные или не интересные)
excl_films = [335977, 799583, 882569, 800158, 872585, 696506, 1029955, 693134, 549509, 956842, 877817]

# предварительные фильтры (по имеющимся параметрам в фильмографии - чтобы дальше не тратить запросы на детализацию)
vote_average_min = 1
vote_count_min = 1
release_date_min = '2026-01-01'
language = 'ru' # добавлять русскояз названия если есть в прокате 

In [4]:
# ПОЛУЧАЕМ ИНФУ ИЗ TMDB + OMDB (РАБОТА С API)
flt_dict = {'vote_average_min' : vote_average_min, 
            'vote_count_min' : vote_count_min,
            'release_date_min' : release_date_min,
            'language' : language
           }

# получаем инфу по списку актеров
df1 = ht.get_actors_filmo(api_key_tmdb, actor_dict, excl_films = excl_films, flt_dict = flt_dict)
# получаем инфу по списку режиссеров
df2 = ht.get_directors_filmo(api_key_tmdb, director_dict, excl_films = np.append(df1.movie_id.values, excl_films), flt_dict = flt_dict)
# итого
df = pd.concat([df1, df2], ignore_index = True)
df_final = ht.get_imdb_info(api_key_omdb, df)

In [5]:
# # ПОСТ ФИЛЬТРАЦИЯ И СОРТИРОВКИ ИЗ ДОП УСЛОВИЙ (НЕТ РАБОТЫ С API)
post_excl_films = []
ht.get_filter(df_final, 
           vote_count_min=1, # vote cnt on tmdb
           runtime_min = 50, # film duration
           vote_average_min = 6, # rating по tmdb
           imdb_rating_min = 6, 
           release_date_min = '2025-10-01',
           genres_flt = ['Animation', 'Documentary'], # жанры которые неинтересно отображать
           excl_films = np.append(excl_films, post_excl_films), # фильмы которые неинтересно отображать
#            drop_columns=['movie_status', 'movie_id', 'countries'] # какие колонки скрыть
           drop_columns=['movie_status', 'countries'] # какие колонки скрыть
          ).sort_values(by='imdb_rating', ascending=False)

,person,person_type,title,original_title,directors,release_date,imdb_rating,awards,genres,runtime,imdb_id,vote_average,vote_count,movie_id
3,pedro-almodovar,director,Горькое Рождество,Amarga Navidad,,2026-03-20,N/A,N/A,"['Drama', 'Comedy']",112,tt28088049,6.388,98,1088548
8,javier-bardem,actor,El ser querido,El ser querido,Rodrigo Sorogoyen,2026-05-16,N/A,N/A,['Drama'],135,tt33451014,7.000,39,1074074
1,robert-pattinson,actor,Одиссея,The Odyssey,Christopher Nolan,2026-07-15,8.3,2 wins total,"['Adventure', 'Action', 'Fantasy']",173,tt33764258,7.707,532,1368337
5,matt-damon,actor,Одиссея,The Odyssey,Christopher Nolan,2026-07-15,8.3,2 wins total,"['Adventure', 'Action', 'Fantasy']",173,tt33764258,7.700,554,1368337
10,edward-norton,actor,Приглашение,The Invite,Olivia Wilde,2026-06-25,7.8,2 wins & 5 nominations total,"['Drama', 'Comedy']",107,tt14173636,7.817,71,950028
11,cristian-mungiu,director,Фьорд,Fjord,,2026-06-13,7.8,2 nominations total,['Drama'],146,tt35410859,8.000,2,1401459
13,cillian-murphy,actor,28 лет спустя: Храм костей,28 Years Later: The Bone Temple,Nia DaCosta,2026-01-14,7.3,3 nominations total,"['Horror', 'Thriller', 'Science Fiction']",109,tt32141377,7.114,1442,1272837
2,robert-pattinson,actor,Вот это драма!,The Drama,Kristoffer Borgli,2026-04-01,7.2,7 nominations total,"['Romance', 'Comedy', 'Drama']",105,tt33071426,6.915,1260,1325734
4,michael-fassbender,actor,Надежда,호프,Na Hong-jin,2026-07-15,7.0,1 nomination total,"['Science Fiction', 'Mystery', 'Action']",157,tt27369017,8.100,10,1058424
6,matt-damon,actor,Лакомый кусок,The Rip,Joe Carnahan,2026-01-13,6.8,N/A,"['Action', 'Thriller', 'Crime']",113,tt32642706,7.084,1755,1306368


### Spotify

In [29]:
# get id + secret on your : https://developer.spotify.com/dashboard/<SPOTIFY_CLIENT_ID>
client_id = ''
client_secret = ''

In [15]:
# посмотреть айдишки наиболее популярных исполнителей по имени
artist_name = 'massive'
top_cnt = 5
ht.get_top_n_id_by_name(client_id, client_secret, artist_name, top_cnt)

['Massive Attack - 6FXMGgJwohJLUSr5nVlf9X - popularity: 71',
 'Hang Massive - 6bkF6GDcyXZn2T0D5Fwldl - popularity: 48',
 'Scratch Massive - 5udXThiZTVoHa4g0GDjgxA - popularity: 42',
 'Massivebass - 5p9GQuGCJcfovjMcDk3ZyI - popularity: 38',
 'Massive B - 36fJ2Mx3ktclhSlBbsUbFY - popularity: 41']

In [10]:
artist_list = [
                '132sZpCaM8ie6byAEcOcRs', # Haelos
               '2VYQTNDsvvKN9wmU5W7xpj', # manson
                '7Ln80lUS6He07XvHI8qqHH', # arctic monk
                '2NPduAUeLVsfIauhRwuft1', # nightwish
#                 '5nPOO9iTcrs9k6yFffPxjH', # Röyksopp
#                 '2a5G7JLmVJNjfFNg8rwLcP', # Till Lindemann
              '3w4VAlllkAWI6m0AV0Gn6a', # hurts,
               '4njdEjTnLfcGImKZu1iSrz', # awolnation
               '3Bd1cgCjtCI32PYvDC3ynO', # london gram
               '3XHO7cRUPCLOr6jwp8vsx5', # alt j
               '163tK9Wjr9P9DmM0AVK7lm', # lorde
               '3iOvXCl6edW5Um0fXEBRXy', # xx
              '0RqtSIYZmd4fiBKVFqyIqD', # 30 second
               '5Pwc4xIPtQLFEnJriah9YJ', # one republ
              '1WgXqy2Dd70QQOU7Ay074N', # aurora,
                '4Z8W4fKeB5YxbusRsdQVPb', # radiohead
               '0BEI7i5sgUuivcfwXLzFmM', # serj tank
               '3wRt3iJpZDOg73CTUkfv5C', # svrcina
#                 '0YCqg8X5gx0MremW64Ud1h', # ддт
                '6mdiAmATAx73kdxrNrnlao', # iron m
                '2ye2Wgw4gimLv2eAKyk1NB', # metallica
                '1moxjboGR7GNWYIMWsRjgG', # florence + m
                '1AZ30JnvQU1pbX6sbRE0Yn', # poets of the fall
                '2N0vFuOoMtAQfBmhsRo24e', # kalanda
                '4Z8W4fKeB5YxbusRsdQVPb', # radiohead
                '1Ffb6ejR6Fe5IamqA5oRUF', # bring me the horizon
                '00FQb4jTyendYWaN8pK0wa', # lana del ray
                '49qiE8dj4JuNdpYGRPdKbF', # stone sour
                '6wWVKhxIU2cEi0K81v7HvP', # rammstein
                '5nGIFgo0shDenQYSE0Sn7c', # Evanescence
                '6XyY86QOPPrYVGvF9ch6wz', # linkin park
                '53KwLdlmrlCelAZMaLVZqU', # james blake
                '6FXMGgJwohJLUSr5nVlf9X', # massive attack

                '6nxDkvGl2oyp6XpSFFZ89s', # portishead
                '1YSI7NofR3G6oM07i31K09', # archive
                '2yIat0oYv4pY6CFrAByV7p', # woodkid
                '56Y9pUv2989SUnP7fX8S7G', # sevdaliza
                '4m66TCOBeUsh9Y679q9Y6Y', # son lux
                '12ChZ9vBvYIiAFMG00pY9O', # muse
                '5u7v98ib77oUOYOBrvGrge', # tame impala
                '3C1SndVvYvRw6mCGO0mZrm', # chvrches
                '5eHT9Un6B3p6vAV8z6Zq9s', # system of a down
                '05fGUMSTnwsYgiZpHvsqtM', # slipknot
                '6Ghvu1oVhOSpMB3pYvYpS4', # deftones
                '3Y7XIsS9pZ9oKt899Y6pS4', # within temptation
                '1mvvUvBTvof9S87vC9vLYm', # ghost
                '1S9SreA90qV9xQZ6Yq6Z7G', # eivør
                '0S0vWE7riWyBrU198vU9Ay', # wardruna
                '7hy0t94uMAnO989OpgXpS4', # heilung
                '7S9SreA90qV9xQZ6Yq6Z7G'  # chelsea wolfe
]

artist_list = list(np.unique(artist_list))

df_spotify = ht.sp_get_albums_info(client_id, client_secret, artist_list)

HTTP Error for GET to https://api.spotify.com/v1/artists/00FQb4jTyendYWaN8pK0wa/albums with Params: {'include_groups': 'album', 'country': None, 'limit': 50, 'offset': 0} returned 403 due to Active premium subscription required for the owner of the app. When the subscription status changes, it can take a few hours before requests are allowed again.


EXCEPTION = 00FQb4jTyendYWaN8pK0wa


UnboundLocalError: cannot access local variable 'tmp' where it is not associated with a value

In [7]:
excl_album_list = ['The Greatest Love', 'From Zero', 'One Assassination Under God - Chapter 1', 
                    'What Happened To The Heart?', 
                   'From Zero: A Cappellas', 'Papercuts: Instrumentals', 'The Phantom Five', '1200 Beats Per Minute']

In [8]:
# ПОСТ ФИЛЬТРАЦИЯ (АПИ НЕ РАБОТАЕТ ЗДЕСЬ)
res = ht.sp_get_filter(df_spotify, 
                release_date_min = '2025-01-01', 
                release_date_max = '2030-01-01',
#                 album_flt = ['deluxe', 'live', 'edition', 'soundtrack', 'remix'], # исключаем слова в названии альбомов
                album_flt = [  'edition', 'soundtrack', 'remix', 'deluxe'], # исключаем слова в названии альбомов
                album_type_flt = ['single'], # убрать эти типы
                max_artists_in_album = 1,
                other_album_author = False, # false когда не хотим чтобы это был фит артиста в альбом другого чувака
#                  columns_excl = ['uri', 'album_type'] # какие колонки не отображать здесь
                 columns_excl = [] # какие колонки не отображать здесь
                 )

res[~res.album.isin(excl_album_list)]

,artist,album,release_date,tracks_cnt,uri,album_artists,album_type
0,Serj Tankian,"Covers, Collaborations & Collages",2025-10-24,10,spotify:album:5ajLhbEaxpUHlzEWtewmiW,['Serj Tankian'],album
1,Radiohead,Hail to the Thief (Live Recordings 2003-2009),2025-08-13,12,spotify:album:5BLrEOEDKoDDg5T8PzdIHN,['Radiohead'],album
2,Lorde,Virgin,2025-06-27,11,spotify:album:28bHj2enHkHVFLwuWmkwlQ,['Lorde'],album
3,Florence + The Machine,Everybody Scream (Chamber Version),2025-11-03,16,spotify:album:1OlrQBbVBbFHZX1YcLC5aZ,['Florence + The Machine'],album
4,Florence + The Machine,Everybody Scream,2025-10-31,12,spotify:album:0z7l9VEJyFMv8p8wffRDaF,['Florence + The Machine'],album
5,Bring Me The Horizon,Lo-files,2025-07-11,23,spotify:album:34pF0wOGswprAZCsI8A1Fs,['Bring Me The Horizon'],album
